# NiyamTrace-X Q1 Experiment 2 — SIL/IIEA Baselines, Ablations, and Safety Properties

**Goal:** supply the component study missing from the paper.

The notebook compares a conventional single-best interpretation against interpretation
intersection, Anchor+intersection, and the full authorization rule. It runs 50,000 formal
scenarios and 20,000 randomized property checks.

This experiment isolates the **authorization mathematics** from LLM extraction quality.

In [ ]:
# Reproducible setup: pin the exact public repository commit audited in the manuscript.
REPO_URL = "https://github.com/bnssaanirudh/NiyamTrace-X.git"
PINNED_COMMIT = "c14661dbd11c42ebd1019b6a1a5c49b8643da137"

!rm -rf /content/NiyamTrace-X
!git clone -q $REPO_URL /content/NiyamTrace-X
%cd /content/NiyamTrace-X
!git checkout -q $PINNED_COMMIT

!pip -q install -e /content/NiyamTrace-X/niyamtrace
!pip -q install pandas numpy scipy scikit-learn matplotlib tqdm statsmodels nbformat

from pathlib import Path
import os, json, math, random, hashlib, statistics, itertools, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path("/content/NiyamTrace-X")
NIYAM = ROOT / "niyamtrace"
RESULTS = Path("/content/niyamtrace_q1_results")
RESULTS.mkdir(exist_ok=True)
print("Pinned commit:", PINNED_COMMIT)
print("Results:", RESULTS)

In [ ]:
SEED = 260911
N_SCENARIOS = 50000
rng = np.random.default_rng(SEED)
EFFECTS = np.arange(24)

def oracle_decision(effect_sets, proposal, anchor_violation=False, policy_permits=True,
                    model_requests_clarify=False, effect_equivalent=False):
    if anchor_violation:
        return "BLOCK"
    sets = [set(s) for s in effect_sets]
    union = set().union(*sets)
    inter = set.intersection(*sets)
    if proposal not in union:
        return "BLOCK"
    if proposal not in inter:
        return "CLARIFY"
    if not policy_permits:
        return "BLOCK"
    # If the semantic model requests clarification despite a common safe effect,
    # suppress it only when all surviving interpretations are effect-equivalent.
    if model_requests_clarify and not effect_equivalent:
        return "CLARIFY"
    return "ALLOW"

def single_best_decision(effect_sets, proposal, anchor_violation=False, policy_permits=True):
    return "ALLOW" if (proposal in set(effect_sets[0]) and policy_permits) else "BLOCK"

def intersection_decision(effect_sets, proposal, anchor_violation=False, policy_permits=True):
    sets = [set(s) for s in effect_sets]
    union = set().union(*sets)
    inter = set.intersection(*sets)
    if proposal not in union:
        return "BLOCK"
    if proposal in inter:
        return "ALLOW" if policy_permits else "BLOCK"
    return "CLARIFY"

def anchor_intersection_decision(effect_sets, proposal, anchor_violation=False, policy_permits=True,
                                 model_requests_clarify=False, effect_equivalent=False):
    if anchor_violation:
        return "BLOCK"
    base = intersection_decision(effect_sets, proposal, False, policy_permits)
    # Ablation without the clarification evidence gate: honor every model clarification.
    if base == "ALLOW" and model_requests_clarify:
        return "CLARIFY"
    return base

def full_decision(effect_sets, proposal, anchor_violation=False, policy_permits=True,
                  model_requests_clarify=False, effect_equivalent=False):
    if anchor_violation:
        return "BLOCK"
    base = intersection_decision(effect_sets, proposal, False, policy_permits)
    if base != "ALLOW":
        return base
    if model_requests_clarify and not effect_equivalent:
        return "CLARIFY"
    return "ALLOW"

def generate_case(i):
    k = int(rng.integers(2,5))
    core_n = int(rng.integers(0,4))
    core = set(rng.choice(EFFECTS, size=core_n, replace=False).tolist()) if core_n else set()
    sets = []
    for _ in range(k):
        pool = sorted(set(EFFECTS.tolist()) - core)
        extra_n = int(rng.integers(0, min(6,len(pool))+1))
        extra = set(rng.choice(pool,size=extra_n,replace=False).tolist()) if extra_n else set()
        sets.append(sorted(core|extra))
    union = set().union(*(set(x) for x in sets))
    if rng.random() < 0.8 and union:
        proposal = int(rng.choice(sorted(union)))
    else:
        outside = sorted(set(EFFECTS.tolist())-union)
        proposal = int(rng.choice(outside if outside else EFFECTS))
    return {
        "case_id":f"formal-{i:06d}", "effect_sets":sets, "proposal":proposal,
        "anchor_violation":bool(rng.random()<0.12),
        "policy_permits":bool(rng.random()<0.92),
        "effect_equivalent":len({tuple(x) for x in sets})==1,
        "model_requests_clarify":bool(rng.random()<0.15),
    }

cases = [generate_case(i) for i in range(N_SCENARIOS)]
print("Generated:", len(cases))

In [ ]:
systems = {
    "single_best": single_best_decision,
    "intersection_only": intersection_decision,
    "anchor_intersection": anchor_intersection_decision,
}
rows = []
for c in cases:
    oracle = oracle_decision(c["effect_sets"], c["proposal"], c["anchor_violation"], c["policy_permits"], c["model_requests_clarify"], c["effect_equivalent"])
    for name, fn in systems.items():
        if name == "anchor_intersection":
            pred = fn(
                c["effect_sets"], c["proposal"], c["anchor_violation"], c["policy_permits"],
                c["model_requests_clarify"], c["effect_equivalent"]
            )
        else:
            pred = fn(c["effect_sets"], c["proposal"], c["anchor_violation"], c["policy_permits"])
        rows.append({"case_id":c["case_id"],"system":name,"oracle":oracle,"pred":pred})
    pred = full_decision(
        c["effect_sets"], c["proposal"], c["anchor_violation"], c["policy_permits"],
        c["model_requests_clarify"], c["effect_equivalent"]
    )
    rows.append({"case_id":c["case_id"],"system":"full","oracle":oracle,"pred":pred})

res = pd.DataFrame(rows)
res["correct"] = res.pred == res.oracle
res["unsafe_allow"] = (res.pred=="ALLOW") & (res.oracle!="ALLOW")
res["false_block"] = (res.pred=="BLOCK") & (res.oracle=="ALLOW")
res["unnecessary_clarify"] = (res.pred=="CLARIFY") & (res.oracle=="ALLOW")

ablation = res.groupby("system").agg(
    n=("case_id","size"),
    accuracy=("correct","mean"),
    unsafe_allow_rate=("unsafe_allow","mean"),
    unsafe_allow_count=("unsafe_allow","sum"),
    false_block_rate=("false_block","mean"),
    unnecessary_clarify_rate=("unnecessary_clarify","mean"),
).reset_index()
display(ablation)

In [ ]:
from scipy.stats import binomtest

wide = res.pivot(index="case_id", columns="system", values="correct")
pairs = []
for other in ["single_best","intersection_only","anchor_intersection"]:
    b = int(((wide["full"]==True)&(wide[other]==False)).sum())
    c = int(((wide["full"]==False)&(wide[other]==True)).sum())
    n = b+c
    p = float(binomtest(min(b,c), n=n, p=0.5, alternative="two-sided").pvalue) if n else 1.0
    pairs.append({"comparison":f"full vs {other}","b_full_correct":b,"c_other_correct":c,"exact_p":p})
pairwise = pd.DataFrame(pairs)
display(pairwise)

In [ ]:
checks = {
    "monotonic_authority": True,
    "no_unsafe_allow_under_intersection": True,
    "anchor_fails_closed": True,
    "clarification_never_broadens": True,
}
counterexamples = []

for i in range(20000):
    c = generate_case(1000000+i)
    sets = [set(s) for s in c["effect_sets"]]
    base_inter = set.intersection(*sets)
    extra = set(rng.choice(EFFECTS,size=int(rng.integers(0,8)),replace=False).tolist())
    new_inter = base_inter & extra
    if not new_inter.issubset(base_inter):
        checks["monotonic_authority"] = False
        counterexamples.append(("monotonic",c))
    pred = intersection_decision(c["effect_sets"],c["proposal"],False,True)
    if pred=="ALLOW" and any(c["proposal"] not in s for s in sets):
        checks["no_unsafe_allow_under_intersection"] = False
        counterexamples.append(("unsafe",c))
    if anchor_intersection_decision(c["effect_sets"],c["proposal"],True,True,False,False)!="BLOCK":
        checks["anchor_fails_closed"] = False
        counterexamples.append(("anchor",c))
    base = intersection_decision(c["effect_sets"],c["proposal"],c["anchor_violation"],c["policy_permits"])
    full = full_decision(c["effect_sets"],c["proposal"],c["anchor_violation"],c["policy_permits"],True,c["effect_equivalent"])
    if base!="ALLOW" and full=="ALLOW":
        checks["clarification_never_broadens"] = False
        counterexamples.append(("clarification",c))

print(json.dumps(checks,indent=2))
print("Counterexamples:",len(counterexamples))
assert all(checks.values()), counterexamples[:1]

In [ ]:
case_df = pd.DataFrame([{
    **{k:v for k,v in c.items() if k!="effect_sets"},
    "effect_sets":json.dumps(c["effect_sets"])
} for c in cases])

case_df.to_csv(RESULTS/"sil_iiea_cases.csv",index=False)
res.to_csv(RESULTS/"sil_iiea_case_results.csv",index=False)
ablation.to_csv(RESULTS/"sil_iiea_ablation.csv",index=False)
pairwise.to_csv(RESULTS/"sil_iiea_pairwise.csv",index=False)
(RESULTS/"sil_iiea_property_tests.json").write_text(json.dumps(checks,indent=2))
(RESULTS/"sil_iiea_table.tex").write_text(
    ablation.to_latex(index=False,float_format=lambda x:f"{x:.6f}")
)

fig, ax = plt.subplots(figsize=(8,4.5))
ax.bar(ablation["system"],ablation["unsafe_allow_rate"])
ax.set_ylabel("Unsafe-Allow rate")
ax.set_title("Formal safety ablation")
ax.tick_params(axis="x",rotation=25)
fig.tight_layout()
fig.savefig(RESULTS/"sil_iiea_unsafe_allow_ablation.png",dpi=220,bbox_inches="tight")
plt.show()

gates = {
    "all_formal_properties_hold":all(checks.values()),
    "full_unsafe_allow_zero":int(ablation.loc[ablation.system=="full","unsafe_allow_count"].iloc[0])==0,
    "full_accuracy_one":float(ablation.loc[ablation.system=="full","accuracy"].iloc[0])==1.0,
}
print(json.dumps(gates,indent=2))
(RESULTS/"sil_iiea_acceptance_gates.json").write_text(json.dumps(gates,indent=2))

In [ ]:
import zipfile
zip_path=Path("/content/NTX_Q1_02_SIL_IIEA_ABLATION_RESULTS.zip")
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for p in RESULTS.glob("sil_iiea_*"):
        z.write(p,arcname=p.name)
print(zip_path)

In [ ]:
# DOWNLOAD RESULTS ZIP
from pathlib import Path
from google.colab import files

download_zip = Path("/content/NTX_Q1_02_SIL_IIEA_ABLATION_RESULTS.zip")

if not download_zip.exists():
    raise FileNotFoundError(
        f"{download_zip} was not found. Run the result-export/ZIP cell above first."
    )

print(f"Downloading: {download_zip.name}")
files.download(str(download_zip))
